# 04 - Custom schema (meeting notes)

Shipped presets (`qa`, `narrative`, `interview`, `dialogue`, `news`) cover most cases. When they do not, define your own Pydantic schema. The extractor will fill exactly the fields you specify, no more, no less.

This notebook builds a `MeetingNotes` schema with fields tuned for engineering retros — attendees, decisions, action items with owners and due dates, blockers, parking-lot items. Runs offline using `MockProvider` with a hand-written response that matches what a real extractor would emit.

What you will learn:
1. How the schema shape constrains the output.
2. How nested models (`ActionItem`) survive the JSON round-trip.
3. How to recover a typed object from `result.payload`.


## Define the schema

In [ ]:
from pydantic import BaseModel, Field

from narrato import Compressor
from narrato.providers.mock import MockProvider


class ActionItem(BaseModel):
    owner: str
    description: str
    due_date: str | None = None
    priority: str | None = None       # e.g. P0 / P1 / P2


class MeetingNotes(BaseModel):
    title: str
    date: str
    attendees: list[str] = Field(default_factory=list)
    decisions: list[str] = Field(default_factory=list)
    action_items: list[ActionItem] = Field(default_factory=list)
    blockers: list[str] = Field(default_factory=list)
    parking_lot: list[str] = Field(default_factory=list)
    next_meeting: str | None = None

print("schema:", MeetingNotes.model_json_schema()["title"])


## The source — an engineering retro


In [ ]:
SOURCE = '''\
Sprint 27 retro, March 21 2026.

Attendees: Priya (eng lead), Alex (PM), Sam (data), Devi (backend), Hugo (frontend), Lin (SRE).

What went well:
- Onboarding screens shipped on time. Devi noted the QA pass found zero regressions.
- We reduced p99 search latency from 980ms to 410ms after the index restructure.

What did not:
- Project Halibut is two weeks behind. The new ranking algorithm has higher accuracy but is
  6x slower than the old one and we have not solved the latency yet.
- Two production incidents related to the new feature flag service. Lin opened a remediation
  document.

Decisions:
- Halibut will hold its current scope; we will not extend the deadline. Priya is reassigning
  Devi from the AI-assistant project to Halibut for two weeks.
- We will adopt a Mondays-only deploy window for the feature flag service until the
  remediation lands.
- Sam will own the new latency dashboard for Halibut.

Action items:
- Devi: profile the ranking algorithm and propose three latency wins by March 26. Priority P0.
- Lin: complete the feature-flag remediation document and circulate by March 24. P1.
- Sam: ship the Halibut latency dashboard by March 28. P1.
- Alex: communicate the deploy-window change to all teams by EOD today.
- Hugo: investigate the intermittent 502s on the public marketing site. P2, due April 4.

Blockers raised:
- Devi: waiting on the GPU pool reservation for ranking benchmarks. Lin will follow up with infra.

Parking lot:
- Should we adopt protobuf v4 for internal RPC? Revisit next sprint.
- Quarterly hackathon idea: a markdown-to-Figma converter.

Next retro: April 4.
'''

print(f"{len(SOURCE)} chars")


## Hand-written canned response

In a real run, `extractor_model="gpt-4o-mini"` or `claude-haiku-4-5` does this for you in one call. Here we hard-code what a competent extractor would return, to keep the notebook offline.


In [ ]:
CANNED = {
    "title": "Sprint 27 retro",
    "date": "March 21 2026",
    "attendees": ["Priya", "Alex", "Sam", "Devi", "Hugo", "Lin"],
    "decisions": [
        "Halibut scope held; deadline not extended.",
        "Devi reassigned from AI assistant to Halibut for 2 weeks.",
        "Mondays-only deploy window for the feature-flag service until remediation lands.",
        "Sam owns the new Halibut latency dashboard.",
    ],
    "action_items": [
        {"owner": "Devi", "description": "Profile ranking algorithm and propose three latency wins.", "due_date": "March 26", "priority": "P0"},
        {"owner": "Lin", "description": "Complete the feature-flag remediation document and circulate.", "due_date": "March 24", "priority": "P1"},
        {"owner": "Sam", "description": "Ship the Halibut latency dashboard.", "due_date": "March 28", "priority": "P1"},
        {"owner": "Alex", "description": "Communicate the deploy-window change to all teams.", "due_date": "EOD today", "priority": None},
        {"owner": "Hugo", "description": "Investigate intermittent 502s on the public marketing site.", "due_date": "April 4", "priority": "P2"},
    ],
    "blockers": ["Devi blocked on GPU pool reservation; Lin to follow up with infra."],
    "parking_lot": [
        "Adopt protobuf v4 for internal RPC?",
        "Hackathon idea: markdown-to-Figma converter.",
    ],
    "next_meeting": "April 4",
}


## Compress with the custom schema

In [ ]:
mock = MockProvider(payload=CANNED)

c = Compressor(
    provider=mock,
    source_lang="en",
    layers=["preprocess", "codebook", "extract"],
    schema=MeetingNotes,        # pass the class directly
)

result = c.compress(SOURCE)

print("tokens in :", result.stats["input_tokens"])
print("tokens out:", result.stats["output_tokens"])
print("ratio    :", round(result.stats["ratio"], 3))
print("valid    :", result.stats["extract"]["valid"])
print("error    :", result.stats["extract"]["validation_error"])


## Recover a typed object

In [ ]:
import json

payload_dict = json.loads(result.payload)
notes = MeetingNotes.model_validate(payload_dict)

print(f"{notes.title} — {notes.date}")
print(f"attendees: {', '.join(notes.attendees)}")
print()
print("action items by owner:")
for item in notes.action_items:
    pri = f"[{item.priority}]" if item.priority else ""
    print(f"  {item.owner:<6} {pri:<4} {item.description}  (due {item.due_date})")


## What this shows

- The extractor returned a JSON object that exactly matches `MeetingNotes`. No extra fields, no missing required fields.
- Nested models survive the round-trip — each `action_items` entry parsed back into an `ActionItem`.
- `result.payload` is always a JSON string; use `MySchema.model_validate(json.loads(result.payload))` to recover the typed object for downstream code.
- The same flow works with any provider — swap `provider=mock` for `provider="openai"` and run with a real key.
